# GIADA Task 34 — correnti e stati come probe privilegiati
Audit diagnostico v3 **senza training né selezione** sui 16 casi già aperti della Task 33 v3. Conserva la fase oracle corretta in v2 e aggiunge il contrasto modello–formula sotto lo stesso V imposto: gli scarti formula–stato nativo sono riportati separatamente. I bracci con V o stati del teacher non sono candidati deployabili. Il report non è una conferma indipendente e non misura speedup.

In [ ]:
from pathlib import Path
import json, subprocess, sys
WORK=Path('/kaggle/working/giada_task34')
REPO=WORK/'giada'
if not REPO.exists():
    subprocess.run(['git','clone','--quiet','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','--quiet','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
sys.path.insert(0,str(REPO))
from src.giada_teacher.task34_privileged_current_state_probes import contract
spec,parent,cfg,t32cfg=contract(REPO)
print({'revision':REVISION,'parent_revision':parent['code_revision'],'cases':parent['confirmation_case_count'],'frozen_families':spec['frozen_families'],'seeds':spec['frozen_seeds']})


In [ ]:
import torch
assert torch.cuda.is_available(), 'Task34 richiede CUDA per i checkpoint congelati.'
OUTPUT=Path('/kaggle/working/artifacts/giada_task34_privileged_current_state_probes')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Avvia una sessione pulita.'
command=[sys.executable,str(REPO/'scripts/run_task34_privileged_current_state_probes.py'),'--output',str(OUTPUT)]
process=subprocess.run(command,check=False)
path=OUTPUT/('final_report.json' if process.returncode==0 else 'failure_report.json')
report=json.loads(path.read_text(encoding='utf-8'))
print({'exit':process.returncode,'valid':report.get('valid'),'diagnostic_complete':report.get('diagnostic_complete'),'cases':report.get('case_count'),'error':report.get('error')})
if process.returncode: raise RuntimeError('Task34 interrotta tecnicamente; leggere failure_report.json, nessun verdetto scientifico.')
assert report['valid'] and report['diagnostic_complete'] and not report['teacher_privileged_arms_selection_eligible']
